<a href="https://colab.research.google.com/github/Jatin94500/ward_pulse/blob/main/dataset_fetures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Mount Google Drive
We will mount your Google Drive to save the downloaded datasets permanently.

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Download Data from GitHub
We will use `git` to clone the repository, filter or copy the specific folder (`p3-ward-civic-demand`), and save it to your Google Drive in a folder named `ward-civic-demand`.

In [7]:
import os
import shutil

target_dir = '/content/drive/My Drive/ward_data'
os.makedirs(target_dir, exist_ok=True)

!git clone https://github.com/CodewithJha/hack-day-lucknow-datasets.git /tmp/hack-day-lucknow-datasets

src_folder = '/tmp/hack-day-lucknow-datasets/p3-ward-civic-demand'

if os.path.exists(src_folder):
    for item in os.listdir(src_folder):
        s = os.path.join(src_folder, item)
        d = os.path.join(target_dir, item)
        if os.path.isdir(s):
            shutil.copytree(s, d, dirs_exist_ok=True)
        else:
            shutil.copy2(s, d)
    print(f"All files copied successfully to: {target_dir}")
else:
    print("Error: Source folder not found in the cloned repository.")

shutil.rmtree('/tmp/hack-day-lucknow-datasets', ignore_errors=True)

Cloning into '/tmp/hack-day-lucknow-datasets'...
remote: Enumerating objects: 2117, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 2117 (delta 2), reused 5 (delta 2), pack-reused 2110 (from 1)
Receiving objects: 100% (2117/2117), 127.13 MiB | 32.02 MiB/s, done.
Resolving deltas: 100% (9/9), done.
All files copied successfully to: /content/drive/My Drive/ward_data


In [9]:
import pandas as pd
import numpy as np
import geopandas as gpd

DATA = "/content/drive/My Drive/ward_data"
DROP_EARLY = True

g = pd.read_csv(f"{DATA}/grievances_public_wards.csv", parse_dates=["grievance_time"])
wf = pd.read_csv(f"{DATA}/ward_features.csv")
grid = pd.read_csv(f"{DATA}/prediction_grid_template.csv")
wards = gpd.read_file(f"{DATA}/wards.geojson")

for d in (g, wf, grid, wards):
    d["ward_id"] = d["ward_id"].astype(str)

known_wards = sorted(g.ward_id.unique())
unknown_wards = sorted(grid.ward_id.unique())
print(len(known_wards), "known wards |", len(unknown_wards), "unknown wards")
print("categories:", sorted(g.category_group.unique()))

156 known wards | 40 unknown wards
categories: ['animals_health', 'electrical', 'other', 'roads', 'solid_waste', 'trees']


In [10]:
g["month"] = g.grievance_time.dt.strftime("%Y-%m")
cats   = sorted(g.category_group.unique())
months = pd.period_range("2020-02", "2025-06", freq="M").strftime("%Y-%m")

counts = (g.groupby(["ward_id", "month", "category_group"])
            .size().rename("count").reset_index())

full = pd.MultiIndex.from_product(
    [known_wards, months, cats],
    names=["ward_id", "month", "category_group"]).to_frame(index=False)

train = full.merge(counts, how="left")
train["count"] = train["count"].fillna(0).astype(int)

if DROP_EARLY:
    first = g.groupby("ward_id").month.min().rename("first_month")
    train = train.merge(first, on="ward_id")
    train = train[train.month >= train.first_month].drop(columns="first_month")

print(train.shape, "| share of zero rows:", round((train["count"] == 0).mean(), 3))
display(train.head())

(60840, 4) | share of zero rows: 0.17


,ward_id,month,category_group,count
0,1,2020-02,animals_health,6
1,1,2020-02,electrical,9
2,1,2020-02,other,2
3,1,2020-02,roads,10
4,1,2020-02,solid_waste,6


In [15]:
pop = wf.set_index("ward_id")["population_2011"]
train["pop"] = train.ward_id.map(pop)
train["rate"] = train["count"] / train["pop"] * 1000

city = (train.groupby(["month", "category_group"])[["count", "pop"]].sum()
             .assign(city_rate=lambda d: d["count"] / d["pop"] * 1000)
             [["city_rate"]].reset_index())

print(city.shape)   # (390, 3)


(390, 3)


In [16]:
wards = wards[["ward_id", "geometry"]].copy()
buf = wards.copy()
buf["geometry"] = buf.geometry.buffer(1e-4)   # ~11 m, so touching wards are caught

pairs = gpd.sjoin(buf, wards, predicate="intersects")[["ward_id_left", "ward_id_right"]]
adj = (pairs[pairs.ward_id_left != pairs.ward_id_right]
       .rename(columns={"ward_id_left": "ward_id", "ward_id_right": "nbr"})
       .drop_duplicates())

# second ring: neighbours of neighbours (excluding self and first ring)
adj2 = (adj.merge(adj, left_on="nbr", right_on="ward_id", suffixes=("", "_x"))
           [["ward_id", "nbr_x"]].rename(columns={"nbr_x": "nbr"}))
adj2 = adj2[adj2.ward_id != adj2.nbr].drop_duplicates()
adj2 = (adj2.merge(adj, on=["ward_id", "nbr"], how="left", indicator=True)
            .query("_merge == 'left_only'")[["ward_id", "nbr"]])

print("wards in graph:", adj.ward_id.nunique())
print("avg neighbours per ward:", round(adj.groupby("ward_id").size().mean(), 1))

wards in graph: 198
avg neighbours per ward: 5.9


/tmp/ipykernel_1795/2249433225.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  buf["geometry"] = buf.geometry.buffer(1e-4)   # ~11 m, so touching wards are caught


In [18]:
rates = (train[["ward_id", "month", "category_group", "rate"]]
         .rename(columns={"ward_id": "nbr", "rate": "nbr_rate"}))

def nbr_feature(adj_df, name):
    m = adj_df.merge(rates, on="nbr")           # drops neighbours without history
    return (m.groupby(["ward_id", "month", "category_group"])
              .agg(**{name: ("nbr_rate", "mean"), name + "_n": ("nbr", "nunique")})
              .reset_index())

n1 = nbr_feature(adj,  "nbr1_rate")
n2 = nbr_feature(adj2, "nbr2_rate")
print(n1.shape, n2.shape)

(77220, 5) (77220, 5)


In [21]:
KEYS = ["ward_id", "month", "category_group"]

def build_features(df, n1=n1, n2=n2):
    df = df.copy()
    df = df.merge(wf, on="ward_id", how="left")

    # ward ratios
    e = 1e-9
    df["roads_per_sqkm"]     = df.road_length_kms / (df.area_sq_km + e)
    df["lights_per_km_road"] = df.streetlights / (df.road_length_kms + e)
    df["lights_per_1000pop"] = df.streetlights / df.population_2011 * 1000
    df["hh_size"]            = df.population_2011 / (df.households_census_2011 + e)
    df["parks_per_sqkm"]     = df.parks / (df.area_sq_km + e)
    df["park_share"]         = df.park_area_sq_m / (df.area_sq_km * 1e6 + e)
    df["lake_share"]         = df.lake_area_sq_m / (df.area_sq_km * 1e6 + e)
    df["busstops_per_sqkm"]  = df.bus_stops / (df.area_sq_km + e)
    df["schools_per_10k"]    = df.govt_schools / df.population_2011 * 10000
    df["log_pop"]            = np.log1p(df.population_2011)

    # neighbour features (with fallbacks)
    df = df.merge(n1, on=KEYS, how="left").merge(n2, on=KEYS, how="left")
    df = df.merge(city, on=["month", "category_group"], how="left")
    df["nbr_rate"] = df.nbr1_rate.fillna(df.nbr2_rate).fillna(df.city_rate)
    df[["nbr1_rate_n", "nbr2_rate_n"]] = df[["nbr1_rate_n", "nbr2_rate_n"]].fillna(0)

    # time features
    p = pd.PeriodIndex(df.month, freq="M")
    df["year"]      = p.year
    df["month_num"] = p.month
    df["t"]         = (p.year - 2020) * 12 + p.month - 2    # months since Feb 2020

    df["category_group"] = df["category_group"].astype("category")
    df["bbmp_zone_name"] = df["bbmp_zone_name"].astype("category")
    return df

train_df = build_features(train)
grid_df  = build_features(grid[KEYS])

print("train:", train_df.shape, "| grid:", grid_df.shape)
print("NaNs in grid nbr_rate:", grid_df.nbr_rate.isna().sum())
print("unknown wards with 0 first-ring neighbours having data:",
      grid_df[grid_df.nbr1_rate_n == 0].ward_id.nunique())

train_df.to_csv("train_table.csv", index=False)
grid_df.to_csv("grid_table.csv", index=False)

train: (60840, 54) | grid: (4320, 51)
NaNs in grid nbr_rate: 0
unknown wards with 0 first-ring neighbours having data: 0


In [22]:
chk = (grid_df.groupby("ward_id")["nbr1_rate_n"].max()
              .value_counts().sort_index())
print(chk)    # number of wards with 1, 2, 3... known neighbours


nbr1_rate_n
2     2
3     6
4    11
5     9
6     7
7     5
Name: count, dtype: int64


In [23]:
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error as mae

drop = ["ward_id","month","ward_name","assembly_constituency_name","mp_constituency",
        "police_stn","bbmp_division","bbmp_sub_division","count","rate","pop"]
FEATS = [c for c in train_df.columns if c not in drop]

P = dict(objective="poisson", learning_rate=0.03, n_estimators=500, num_leaves=31,
         min_child_samples=50, subsample=0.8, subsample_freq=1,
         colsample_bytree=0.8, verbose=-1)

def make_nbr(visible):
    r = rates[rates.nbr.isin(visible)]
    def f(adj_df, name):
        m = adj_df.merge(r, on="nbr")
        return (m.groupby(["ward_id","month","category_group"])
                  .agg(**{name: ("nbr_rate","mean"), name+"_n": ("nbr","nunique")})
                  .reset_index())
    return f(adj, "nbr1_rate"), f(adj2, "nbr2_rate")

wards_arr = np.array(known_wards)
scores = []
for k, (tr_i, va_i) in enumerate(KFold(5, shuffle=True, random_state=42).split(wards_arr)):
    tr_w, va_w = wards_arr[tr_i], wards_arr[va_i]
    n1f, n2f = make_nbr(tr_w)                    # neighbour info only from visible wards
    dfk = build_features(train, n1f, n2f)
    tr = dfk[dfk.ward_id.isin(tr_w)]
    va = dfk[dfk.ward_id.isin(va_w) & (dfk.month >= "2024-01")]   # same months as the real task

    m = lgb.LGBMRegressor(**P).fit(tr[FEATS], tr["rate"], sample_weight=tr["pop"])
    pred = m.predict(va[FEATS]) * va["population_2011"] / 1000
    base = va["city_rate"] * va["population_2011"] / 1000        # baseline: city average
    scores.append((mae(va["count"], pred), mae(va["count"], base)))
    print(f"fold {k}: model MAE {scores[-1][0]:.2f} | baseline MAE {scores[-1][1]:.2f}")

print("average:", np.mean(scores, axis=0).round(2))


ValueError: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: population_density_2011_persons_sq_km: object, decadal_popn_growth_rate_2011_2001: object, decadal_households_growth_rate_2011_2001: object